In [80]:
from pyscf import gto, scf, cc

a = 2.5 # 2aB
nH = 2
atoms = ""
for i in range(nH):
    atoms += f"H {i*a:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="6-31g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()
# et = mycc.ccsd_t()
# print(mycc.e_tot + et)

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-14-generic', version='#14~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jan 15 15:52:10 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Thu Feb 19 20:42:54 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

(np.float64(-0.046016701781519564),
 array([[-1.19138898e-16,  2.39653420e-02, -7.50791923e-16]]),
 array([[[[-2.40933063e-01, -2.50982495e-15, -6.64737637e-02],
          [-2.50982495e-15, -2.28838911e-02, -3.09632801e-16],
          [-6.64737637e-02, -3.09632801e-16, -3.05175309e-02]]]]))

In [72]:
print(mycc.energy(t1=mycc.t1, t2=mycc.t2*0))

0.00011738856385932526


In [ ]:
ut1 = [mycc.t1, mycc.t1]
ut2 = [mycc.t2, mycc.t2, mycc.t2]
print(myucc.energy(t1=ut1, t2=ut2))

-0.06697999104725762


In [ ]:
np.abs(myucc.t2[1] - myucc.t2[0]).max()

np.float64(2.0816681711721685e-17)

In [ ]:
t2_asy = mycc.t2
t2_asy = (t2_asy - t2_asy.transpose(1,0,2,3)) / 2
t2_asy = (t2_asy - t2_asy.transpose(0,1,3,2)) / 2

In [89]:
ut1 = [mycc.t1, mycc.t1]
ut2 = [t2_asy,
       mycc.t2,
       t2_asy]
myucc = cc.UCCSD(mf,frozen=nfrozen)
myucc.kernel(t1=ut1, t2=ut2)


******** <class 'pyscf.cc.uccsd.UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(1), np.int64(1)), nmo = (4, 4)
frozen orbitals 0
max_cycle = 50
direct = 0
conv_tol = 1e-07
conv_tol_normt = 1e-06
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 2075 MB)
Init E_corr(UCCSD) = -0.0460167017815196
cycle = 1  E_corr(UCCSD) = -0.0460166980579313  dE = 3.72358831e-09  norm(t1,t2) = 2.85125e-08
UCCSD converged
E(UCCSD) = -1.077319488708103  E_corr = -0.04601669805793127


(np.float64(-0.046016698057931266),
 (array([[-9.32613401e-17,  2.39653405e-02, -7.57777170e-16]]),
  array([[-1.13693497e-16,  2.39653405e-02, -7.51780963e-16]])),
 (array([[[[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]]]]),
  array([[[[-2.40933035e-01, -2.50411694e-15, -6.64737615e-02],
           [-2.50724748e-15, -2.28838956e-02, -2.81461022e-16],
           [-6.64737615e-02, -3.06042521e-16, -3.05175298e-02]]]]),
  array([[[[0., 0., 0.],
           [0., 0., 0.],
           [0., 0., 0.]]]])))

In [88]:
(myucc.t2[1] + myucc.t2[1].transpose(1,0,2,3)) / 2

array([[[[-2.40933010e-01, -2.50179932e-15, -6.64737569e-02],
         [-2.50541690e-15, -2.28838986e-02, -2.72809115e-16],
         [-6.64737569e-02, -3.08663441e-16, -3.05175302e-02]]]])

In [67]:
import numpy as np
np.abs(myucc.t2[0] - myucc.t2[2]).max()

np.float64(1.3877787807814457e-17)

In [34]:
import numpy as np
mycct = cc.CCSDT(mf)
nocc, nmo = mycct.nocc, mycct.nmo
nvir = nmo - nocc
t1 = mycc.t1
t2 = mycc.t2
from math import prod, factorial
nx = lambda n, order: prod(n + i for i in range(order)) // factorial(order)
cc_order = mycct.cc_order
tamps = [t1, t2]
for order in range(2, cc_order - 1):
    t = np.zeros((nocc,) * (order + 1) + (nvir,) * (order + 1), dtype=t1.dtype)
    tamps.append(t)
if mycct.do_tri_max_t:
    t = np.zeros((nx(nocc, cc_order),) + (nvir,) * cc_order, dtype=t1.dtype)
else:
    t = np.zeros((nocc,) * cc_order + (nvir,) * cc_order, dtype=t1.dtype)
tamps.append(t)

#mycct = cc.CCSDT(mf)#, compact_tamps=True)
mycct.conv_tol = 1e-7
mycct.conv_tol_normt = 1e-6
mycct.level_shift = 0.2
mycct.verbose = 4
mycct.blksize = 1
mycct.set_einsum_backend('pyscf')
mycct.incore_complete = False
print(mycct.ao2mo)
mycct.kernel(tamps=tamps)

print("CCSDT Energy: ", mycct.e_tot)

<bound method _ao2mo_rcc of <pyscf.cc.rccsdt.RCCSDT object at 0x7637b406a650>>

******** <class 'pyscf.cc.rccsdt.RCCSDT'> ********
RCCSDT nocc = 3, nmo = 12
Allocating only the i<=j<=k part of the T3 amplitude in memory
max_cycle = 50
conv_tol = 1e-07
conv_tol_normt = 1e-06
diis with the T3 amplitude
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 205 MB)
einsum_backend: pyscf
_make_eris_incore_RCCSDT
blksize  1    blksize_oovv  3    blksize_oooo  3    blksize_vvvv  1

WARN: A large `blksize_oovv` is being used, which may cause large memory consumption
      for storing contraction intermediates. If memory is sufficient, consider using
      `pyscf.cc.rccsdt_highm.RCCSDT` instead.


WARN: A large `blksize_oooo` is being used, which may cause large memory consumption
      for storing contraction intermediates. If memory is sufficient, consider using
      `pyscf.cc.rccsdt_highm.RCCSDT` instead.

Approximate memory usage estimate
    T3

In [35]:
mycct.blksize

1

In [41]:
from pyscf import lib
from pyscf.cc import rccsdt

blksize = 3 #mycct.blksize
nvir = mycct.nmo - mycct.nocc
t3 = mycct.t3
t3_tmp = np.empty((blksize,) * 3 + (nvir,) * 3, dtype=t3.dtype)
# rccsdt._unpack_t3_(mycct, t3_tmp, )
for k0, k1 in lib.prange(0, nocc, blksize):
    bk = k1 - k0
    for j0, j1 in lib.prange(0, nocc, blksize):
        bj = j1 - j0
        for i0, i1 in lib.prange(0, nocc, blksize):
            bi = i1 - i0
            print(i0, i1, j0, j1, k0, k1)

nocc = mycct.nocc
t3_tmp = rccsdt._unpack_t3_(mycct, t3, t3_tmp, 0, nocc, 0, nocc, 0, nocc)

0 3 0 3 0 3


In [43]:
t3_tmp.shape

(3, 3, 3, 9, 9, 9)

In [46]:
# example for PT2
options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 100,
            'seed': 2,
            'walker_type': 'rhf',
            'trial': 'ccsd_pt2',
            'dt':0.005,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode': None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (3, 3)
# Number of basis functions: 12
# Number of Cholesky vectors: 36
#


In [47]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
# ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
# ept = jnp.array(jnp.sum(ept_sp) / prop.n_walkers)
prop_data["e_estimate"] = e_init
# eci = trial.calc_energy(
#     prop_data['walkers'], ham_data, wave_data)
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# norb: 12
# nelec: (3, 3)
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 100
# seed: 2
# walker_type: rhf
# trial: ccsd_pt2
# dt: 0.005
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# symmetry: False
# save_walkers: False
# n_batch: 1
# max_error: 0.001
-3.0908772855484976
-4.5114160496240174e-07


In [ ]:
from jax import jit

@partial(jit, static_argnums=0)
def _calc_overlap_ci3(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ci3: jax.Array
    ) -> complex:
    noccA = self.nelec[0]
    noccB = self.nelec[1]
    ci3AAA, ci3AAB, ci3ABB, ci3BBB = ci3

    green_a = (walker_up.dot(jnp.linalg.inv(walker_up[: walker_up.shape[1], :]))).T
    green_b = (walker_dn.dot(jnp.linalg.inv(walker_dn[: walker_dn.shape[1], :]))).T
    green_a, green_b = green_a[:, noccA:], green_b[:, noccB:]
    # o0 = jnp.linalg.det(walker_up[:noccA, :]) * jnp.linalg.det(walker_dn[:noccB, :])
    # o1 = jnp.einsum("ia,ia", ci1A, green_a) + jnp.einsum("ia,ia", ci1B, green_b)
    # o2 = (
    #     0.5 * jnp.einsum("iajb, ia, jb", ci2AA, green_a, green_a)
    #     + 0.5 * jnp.einsum("iajb, ia, jb", ci2BB, green_b, green_b)
    #     + jnp.einsum("iajb, ia, jb", ci2AB, green_a, green_b)
    # )

    # ci3AAA = wave_data["ci3AAA"]
    # ci3AAB = wave_data["ci3AAB"]
    # ci3ABB = wave_data["ci3ABB"]
    # ci3BBB = wave_data["ci3BBB"]

    o3 = ( (1/6) * jnp.einsum("iajbkc, ia, jb, kc", ci3AAA, green_a, green_a, green_a)
         + (1/6) * jnp.einsum("iajbkc, ia, jb, kc", ci3BBB, green_b, green_b, green_b)
         + (1/2) * jnp.einsum("iajbkc, ia, jb, kc", ci3AAB, green_a, green_a, green_b)
         + (1/2) * jnp.einsum("iajbkc, ia, jb, kc", ci3ABB, green_a, green_b, green_b) ) 

    return o3